In [ ]:
from Influence_Modelling import XsvNetwork, Network, Sampling, Metrics, Heuristics
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import scienceplots 
plt.style.use(["science", "no-latex"])
import seaborn as sns
import time
enron  =XsvNetwork.XsvNetwork("enron")
net = Network.Network("")
net.set_nx(Sampling.mhda_sample(enron.get_nx(), 4000))

In [ ]:
print(net.get_nx().nodes)

In [ ]:
#sizes = [x for x in range(1, 100, 8)]
sizes = [3, 5, 7]

In [ ]:
deg = [net.run_ICC(Heuristics.double_deg_discount_h(net.get_nx(), size)) for size in sizes]
deg_nets = [de[1] for de in deg]

deg_discount = [net.run_ICC(Heuristics.double_deg_discount_h(net.get_nx(), size)) for size in sizes]
deg_discount_nets = [de[1] for de in deg_discount]

clust = [net.run_ICC(Heuristics.local_clustering_coefficient_h(net.get_nx(), size)) for size in sizes]
clust_nets = [de[1] for de in clust]

betwe = [net.run_ICC(Heuristics.betweenness_centrality_h(net.get_nx(), size)) for size in sizes]
betwe_nets = [de[1] for de in betwe]

In [ ]:
katz= [net.run_ICC(Heuristics.katz_centrality_h(net.get_nx(), size)) for size in sizes]
katz_nets = [de[1] for de in katz]

eigen = [net.run_ICC(Heuristics.katz_centrality_h(net.get_nx(), size)) for size in sizes]
eigen_nets = [de[1] for de in eigen]


In [ ]:
def compute_intersection(G1, G2):
  g1 = set([v for (v, u) in G1.nodes(data = True) if u['status'] != "inactive"])
  g2 = set([v for (v, u) in G2.nodes(data = True) if u['status'] !="inactive"])
  return len(list(set.intersection(g1, g2))) / (0.5 * (len(g1) + len(g2)))

In [ ]:
sum([compute_intersection(deg_nets[i], deg_discount_nets[i]) for i in range(len(sizes))]) / len(sizes)

In [ ]:
def compute_intersection(G1, G2):
    g1 = {v for (v, u) in G1.nodes(data=True) if u['status'] != "inactive"}
    g2 = {v for (v, u) in G2.nodes(data=True) if u['status'] != "inactive"}
    
    inter = g1.intersection(g2)
    score = len(inter) / (0.5 * (len(g1) + len(g2)))
    
    return inter, score

In [ ]:
def pairwise_intersections(graphs):
    names = list(graphs.keys())
    matrix = []

    for g1_name in names:
        row = []
        for g2_name in names:
            _, score = compute_intersection(graphs[g1_name], graphs[g2_name])
            row.append(score)
        matrix.append(row)

    return names, np.array(matrix)

In [ ]:
i = 2
names, matrix = pairwise_intersections({"Discount":deg_discount_nets[i], "Degree":deg_nets[i], "Clustering": clust_nets[i], "Betweenness": betwe_nets[i], "Katz": katz_nets[i], "Eigen": eigen_nets[i]})
fig, ax = plt.subplots()
fig.set_dpi(200)
fig.set_size_inches(5, 5)
sns.heatmap(matrix, annot=True, xticklabels=names, yticklabels=names, mask=np.tril(np.ones_like(matrix, dtype=bool), k=-1), cmap="RdYlGn_r", cbar=False)

In [ ]:
deg_discount_nets[0].nodes(data=True)

In [ ]:
def find_overlaps(methods, size_set, axis, network):
    intersections = {}
    names = []
    for method in methods:
        name = str(method).split(" ")[1]
        names.append(name)
        seeds = set(method(network, size_set))
        intersections[name] = seeds
    abs_matrix = []
    for name in names:
        row = []
        for other_name in names:
            row.append(
                len(
                    list(
                        set.intersection(intersections[name], intersections[other_name])
                    )
                )
            )
        abs_matrix.append(row)
    scaled = np.array(abs_matrix) / size_set
    axis.set_title("Proportion of Selected Seed Size Intersecting (100 Nodes)")
    sns.heatmap(scaled, annot=True, ax=axis, cmap="RdYlGn_r", cbar=False, xticklabels=names, yticklabels=names, mask=np.tril(np.ones_like(scaled, dtype=bool), k=-1))
    axis.set_yticklabels(axis.get_yticklabels(), rotation=0)

In [ ]:
fig, axs = plt.subplots()
fig.set_size_inches(5,5)
fig.set_dpi(200)
find_overlaps([Heuristics.deg_h, Heuristics.deg_discount_h, Heuristics.double_deg_discount_h, Heuristics.mean_neighbour_deg_h, Heuristics.betweenness_centrality_h, Heuristics.local_clustering_coefficient_h], 100, axs, net.get_nx())